In [1]:
!pip install datasets

In [2]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight
from datasets import Dataset
import torch
import re
import json
# from google.colab import drive



In [3]:
!conda install -y gdown 

!gdown --id 1FNWl_JZOh4lKDGVZc8T0Vno9qrdRIVpT


/bin/bash: line 1: conda: command not found
/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1FNWl_JZOh4lKDGVZc8T0Vno9qrdRIVpT
From (redirected): https://drive.google.com/uc?id=1FNWl_JZOh4lKDGVZc8T0Vno9qrdRIVpT&confirm=t&uuid=0796ff50-8d0c-4632-926e-da047a79d032
To: /kaggle/working/Train.csv
100%|█████████████████████████████████████████| 701M/701M [00:03<00:00, 219MB/s]


In [4]:
# Step 1: Load the dataset from Google Drive
# drive.mount('/content/drive')
# dataset_path = '/content/drive/MyDrive/Datasets/Train.csv'
data = pd.read_csv('./Train.csv')
data.head()




,Unnamed: 0,text,genre,label,label_model,text_cleaned
0,0,"It starts with pain, followed by hate\nFueled ...",rock,9,LABEL_9,"It starts with pain, followed by hate\nFueled ..."
1,1,Freedom!\nAlone again again alone\nPatiently w...,rock,9,LABEL_9,Freedom!\nAlone again again alone\nPatiently w...
2,2,"Biting the hand that feeds you, lying to the v...",rock,9,LABEL_9,"Biting the hand that feeds you, lying to the v..."
3,3,You say you know just who I am\nBut you can't ...,rock,9,LABEL_9,You say you know just who I am\nBut you can't ...
4,4,My heart is beating faster can't control these...,rock,9,LABEL_9,My heart is beating faster can't control these...


In [5]:
# Step 2: Data Preprocessing
# Text Cleaning: Remove unnecessary characters, noise, and symbols
def clean_text(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

data['text_cleaned'] = data['text_cleaned'].fillna("").apply(clean_text)


In [6]:
# Optional: Sample a subset of the data (e.g., 100k rows)
sample_size = 150000  # Adjust as needed
data = data.sample(sample_size, random_state=42)



# Prepare cleaned text and target column
texts = data['text_cleaned'].tolist()
labels = data['label_model'].fillna("").tolist()



In [7]:
# Step 3: Label Encoding
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)



In [8]:
# Step 4: Handling Imbalanced Data
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(encoded_labels),
    y=encoded_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels
)



In [9]:
# Step 5: Tokenization and Initialization
model_name = "distilbert-base-uncased"  # A lightweight model for fine-tuning
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding=True, max_length=512)

# Prepare datasets
train_data = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_data = Dataset.from_dict({'text': val_texts, 'label': val_labels})

train_data = train_data.map(tokenize_function, batched=True)
val_data = val_data.map(tokenize_function, batched=True)

# Define metrics for evaluation
def compute_metrics(pred):
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted'),
        'precision': precision_score(labels, predictions, average='weighted'),
        'recall': recall_score(labels, predictions, average='weighted')
    }



Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

In [10]:
pip install --upgrade transformers datasets accelerate


Note: you may need to restart the kernel to use updated packages.


In [11]:
from torch.profiler import profile, ProfilerActivity
import os
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Disable W&B logging if not needed
os.environ["WANDB_DISABLED"] = "true"

def cross_validate_model():
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    # Pre-tokenize the entire dataset once to avoid redundant tokenization
    tokenized_dataset = Dataset.from_dict({'text': texts, 'label': encoded_labels}).map(
        tokenize_function, batched=True
    )

    for fold, (train_idx, val_idx) in enumerate(skf.split(texts, encoded_labels)):
        print(f"Running Fold {fold + 1}/5")

        # Ensure index limits match dataset size to avoid KeyError
        train_idx = train_idx[:min(10000, len(train_idx))]
        val_idx = val_idx[:min(2000, len(val_idx))]

        # Subset training and validation datasets
        train_fold_data = tokenized_dataset.select(train_idx)
        val_fold_data = tokenized_dataset.select(val_idx)

        # Log dataset sizes for debugging
        print(f"Fold {fold + 1}: Train size={len(train_fold_data)}, Val size={len(val_fold_data)}")

        # Load model for the current fold
        fold_model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=len(label_encoder.classes_)
        )

        fold_training_args = TrainingArguments(
            output_dir=f"./results_fold_{fold + 1}",  # Checkpoint directory
            run_name=f"fold_{fold + 1}_experiment",  # Custom run name
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=8,  # Reduced batch size for memory efficiency
            per_device_eval_batch_size=8,
            gradient_accumulation_steps=4,  # Simulate larger effective batch size
            num_train_epochs=2,  # Reduced epochs
            weight_decay=0.01,
            logging_dir=f"./logs_fold_{fold + 1}",
            logging_steps=100,  # Adjusted logging frequency
            save_total_limit=1,  # Save only the best model
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            push_to_hub=False,
        )

        # Define Trainer for the current fold
        trainer = Trainer(
            model=fold_model,
            args=fold_training_args,
            train_dataset=train_fold_data,
            eval_dataset=val_fold_data,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics,
        )

        # Clear GPU cache before training
        torch.cuda.empty_cache()

        try:
            # Train without profiling to avoid overhead
            trainer.train()

            # Evaluate the model on the validation set
            eval_results = trainer.evaluate()
            results.append(eval_results)
        except Exception as e:
            print(f"Error during training or evaluation for fold {fold + 1}: {e}")
            results.append({"error": str(e)})

    return results


# Automatically run cross-validation
try:
    cv_results = cross_validate_model()
    print("Cross-validation results:", cv_results)
except Exception as e:
    print("Error in cross-validation:", str(e))


Map:   0%|          | 0/150000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Running Fold 1/5
Fold 1: Train size=10000, Val size=2000


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-11-464c8ac661b4>:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.101500,1.178720,0.593500,0.545059,0.551881,0.593500


...

Running Fold 5/5
Fold 5: Train size=10000, Val size=2000


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.141400,1.125796,0.604500,0.564487,0.578162,0.604500


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Cross-validation results: [{'eval_loss': 1.1787199974060059, 'eval_accuracy': 0.5935, 'eval_f1': 0.5450590679256632, 'eval_precision': 0.5518812725002468, 'eval_recall': 0.5935, 'eval_runtime': 16.4231, 'eval_samples_per_second': 121.779, 'eval_steps_per_second': 15.222, 'epoch': 1.9952}, {'eval_loss': 1.1514261960983276, 'eval_accuracy': 0.609, 'eval_f1': 0.5680009465334122, 'eval_precision': 0.5666620375598694, 'eval_recall': 0.609, 'eval_runtime': 16.2844, 'eval_samples_per_second': 122.817, 'eval_steps_per_second': 15.352, 'epoch': 1.9952}, {'eval_loss': 1.185078501701355, 'eval_accuracy': 0.5905, 'eval_f1': 0.5401327281749133, 'eval_precision': 0.540117967137227, 'eval_recall': 0.5905, 'eval_runtime': 16.3986, 'eval_samples_per_second': 121.961, 'eval_steps_per_second': 15.245, 'epoch': 1.9952}, {'eval_loss': 1.1718670129776, 'eval_accuracy': 0.599, 'eval_f1': 0.5558793096668121, 'eval_precision': 0.5681556197773048, 'eval_recall': 0.599, 'eval_runtime': 16.3486, 'eval_samples_per

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [12]:
# Load the TensorBoard extension
%load_ext tensorboard

# Start TensorBoard to visualize logs
%tensorboard --logdir=./logs

<IPython.core.display.Javascript object>

In [13]:
pip install --upgrade transformers accelerate deepspeed


Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
import json
from accelerate import Accelerator
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

# Ensure DeepSpeed is disabled
os.environ["ACCELERATE_USE_DEEPSPEED"] = "false"  # Disable DeepSpeed
os.environ["ACCELERATE_DISABLE_RICH"] = "1"  # Optional: Disable rich logging

# Initialize Accelerator
accelerator = Accelerator()

# Load the model (using distilbert-base-uncased)
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=len(label_encoder.classes_)
)

# Enable gradient checkpointing for the model
model.gradient_checkpointing_enable()

# Define training arguments without DeepSpeed
training_args = TrainingArguments(
    output_dir="./final_results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./final_logs",
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    fp16=True,
    gradient_accumulation_steps=4,
    lr_scheduler_type="linear",
    warmup_steps=100,
    report_to="none",  # Disable all reporting
)

# Prepare the Trainer with Accelerator
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # Early stopping
)

# Cache tokenized datasets (for faster training)
def cache_tokenization(dataset, tokenizer):
    return dataset.map(lambda x: tokenizer(x['text'], truncation=True, padding=True), batched=True)

train_data = cache_tokenization(train_data, tokenizer)
val_data = cache_tokenization(val_data, tokenizer)

# Prepare everything for training with Accelerator
train_data, val_data, model, trainer.optimizer = accelerator.prepare(
    train_data, val_data, model, trainer.optimizer
)

# Train the model using Trainer's train method
trainer.train()

# Save the fine-tuned model
trainer.save_model("./fine_tuned_model")

# Save the label encoder mapping
label_encoder_mapping = dict(zip(range(len(label_encoder.classes_)), label_encoder.classes_))
with open("./fine_tuned_model/label_mapping.json", "w") as f:
    json.dump(label_encoder_mapping, f)

print("Model fine-tuned and saved successfully.")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-15-5e2a677ca708>:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.875600,1.031873,0.648167,0.615539,0.616382,0.648167
2,0.896600,0.992772,0.662767,0.632747,0.662083,0.662767
3,0.839400,0.997545,0.665400,0.637148,0.661955,0.665400


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Model fine-tuned and saved successfully.


In [36]:
import os
import json
import numpy as np
import zipfile

# Assuming tokenizer, model, and label_encoder are defined elsewhere in your code

import torch

# Step 8: Prediction
def predict_texts(input_files):
    allowed_extensions = [".safetensors", ".json", ".txt"]
    predictions = []

    # Move model to the appropriate device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)

    for file in input_files:
        if not any(file.endswith(ext) for ext in allowed_extensions):
            raise ValueError(f"Unsupported file extension for file: {file}")

        if file.endswith(".json"):
            with open(file, "r") as f:
                data = json.load(f)
            input_texts = data if isinstance(data, list) else [data]
        elif file.endswith(".txt"):
            with open(file, "r") as f:
                input_texts = f.readlines()
            input_texts = [line.strip() for line in input_texts]
        elif file.endswith(".safetensors"):
            raise ValueError("Processing .safetensors files is not supported yet.")

        # Tokenize inputs and move them to the same device as the model
        tokenized_inputs = tokenizer(input_texts, truncation=True, padding=True, max_length=512, return_tensors="pt").to(device)
        
        # Forward pass through the model
        outputs = model(**tokenized_inputs)
        logits = outputs.logits.detach().cpu().numpy()
        file_predictions = np.argmax(logits, axis=1)
        file_predicted_labels = label_encoder.inverse_transform(file_predictions)
        predictions.extend(file_predicted_labels)

    return predictions


In [58]:
import os
import json

# Define the path to the JSON file
json_path = "/kaggle/input/mlds-sample/mlds.json"

# Check if the JSON file exists and load it
if os.path.exists(json_path):
    try:
        with open(json_path, 'r') as file:
            data = json.load(file)  # Load the JSON as a Python object
            print("JSON Loaded Successfully.")
    except json.JSONDecodeError:
        print(f"Failed to decode JSON file: {json_path}")
        data = None
else:
    print(f"File not found: {json_path}")
    data = None

# Extract text fields from the JSON structure
text_data = []
if data:
    if isinstance(data, list):  # Check if the JSON is a list of dictionaries
        for item in data:
            for key, value in item.items():
                # Append both keys and values as strings
                text_data.append(key)
                text_data.append(value)
    elif isinstance(data, dict):  # If the JSON is a single dictionary (unlikely here)
        for key, value in data.items():
            text_data.append(key)
            text_data.append(value)
    else:
        print("Unsupported JSON structure.")
else:
    print("No data to process.")

# Print the extracted text data
print(f"Extracted Text Data: {text_data}")




JSON Loaded Successfully.
Extracted Text Data: ["It starts with pain, followed by hate\nFueled by the endless questions no one can answer\nA stain covers your heart and tears you apart\nJust like a sleeping cancer\nI don't believe men are born to be killers\nI don't believe the world can be saved\nHow did you get here and when did it start?\nAn innocent child with a thorn in his heart\nWhat kind of world do we live in?\nWhere love is divided by hate\nLoosing control of our feelings\nWe all must be dreaming this life away\nIn a world so cold\nAre you sane, where's the shame?\nA moment of time passes by you cannot rewind\nWho's to blame and where did it start?\nIs there a cure for your sickness\nHave you no heart?\nI don't believe men are born to be killers\nI don't believe the world can't be saved\nHow did you get here and when did it start?\nAn innocent child with a thorn in his heart\nWhat kind of world do we live in?\nWhere love is divided by hate\nSelling our soul for no reason\nWe 

In [59]:
# Example predict_texts function (modified to return only labels)
def predict_texts(texts):
    # Placeholder logic for prediction
    # For demonstration, using fixed labels (replace with your actual prediction logic)
    return [f"Label {i + 1}" for i in range(len(texts))]

# Pass the extracted text to predict_texts
try:
    if isinstance(text_data, list) and all(isinstance(item, str) for item in text_data):
        example_predictions = predict_texts(text_data)  # Pass the list of strings
        print("Predicted Labels Only:", example_predictions)  # Print labels only
    else:
        raise ValueError("Extracted text data is not in the correct format.")
except ValueError as e:
    print(f"Error in prediction: {e}")


Predicted Labels Only: ['Label 1', 'Label 2', 'Label 3', 'Label 4', 'Label 5', 'Label 6', 'Label 7', 'Label 8', 'Label 9', 'Label 10']


In [60]:
import os
import json

# Define the directory path
model_dir = './fine_tuned_model'
os.makedirs(model_dir, exist_ok=True)

# Create a dummy binary file
final_model_path = os.path.join(model_dir, 'final_model.bin')


# Create a dummy JSON file
label_mapping_path = os.path.join(model_dir, 'label_mapping.json')
# dummy_mapping = {"label_1": "Positive", "label_2": "Negative"}
# with open(label_mapping_path, 'w') as f:
#     json.dump(dummy_mapping, f)

print("Files created successfully.")
print(os.listdir(model_dir))  # Verify the created files


Files created successfully.
['tokenizer_config.json', 'config.json', 'vocab.txt', 'model.safetensors', 'label_mapping.json', 'special_tokens_map.json', 'training_args.bin', 'tokenizer.json']


In [61]:
import os
import json
import zipfile

# List files in the working directory
print(os.listdir('/kaggle/working/'))

# Define paths for model files
model_dir = './fine_tuned_model'
final_model_path = os.path.join(model_dir, 'final_model.bin')
label_mapping_path = os.path.join(model_dir, 'label_mapping.json')

# Check if both files exist before zipping
if not os.path.isfile(final_model_path):
    print(f"File not found: {final_model_path}")
if not os.path.isfile(label_mapping_path):
    print(f"File not found: {label_mapping_path}")

# Proceed to zip if both files exist
zip_filename = '/kaggle/working/model_files.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    if os.path.isfile(final_model_path):
        zipf.write(final_model_path, arcname='final_model.bin')
    if os.path.isfile(label_mapping_path):
        zipf.write(label_mapping_path, arcname='label_mapping.json')

print(f"Model files saved to {zip_filename}. You can download it from the output section.")


['.virtual_documents', 'model_files.zip', 'fine_tuned_model', 'results_fold_2', 'results_fold_5', 'logs_fold_4', 'final_results', 'final_logs', 'logs_fold_3', 'results_fold_1', 'Train.csv', 'results_fold_4', 'results_fold_3', 'logs_fold_1', 'logs_fold_2', 'logs_fold_5']
File not found: ./fine_tuned_model/final_model.bin
Model files saved to /kaggle/working/model_files.zip. You can download it from the output section.
